# Clases en Python (class)

Temas: `class`, instancias, atributos, `__init__`, `self`, métodos, `__str__` y `__repr__`.

## lo que ya sabemos hacer

Hasta ahora guardamos una observación del SMN en un diccionario:

In [ ]:
obs = {"ciudad": "Bariloche", "temperatura": 8.5, "humedad": 70, "viento": "Oeste 25"}
obs["temperatura"]

Una **clase** agrupa *datos* (atributos) y *comportamiento* (métodos) en un nuevo tipo.

## Definir una clase y crear instancias

La clase es el molde; cada objeto creado a partir de ella es una **instancia**.

In [4]:
class Observacion:
    pass
    

In [8]:
a = list()
b = tuple()
c = "hola"
isinstance(c, str)

True

In [9]:
isinstance(c, list)

False

In [6]:
o1 = Observacion()
o2 = Observacion()
print(type(o1))
print(o1 is o2)

<class '__main__.Observacion'>
False


Ya usamos clases sin saberlo: `str`, `list`, `dict` son clases y `"hola"` es una instancia de `str`.

In [2]:
print(type("hola"), type([1, 2]), type({}))

isinstance(o1, Observacion)

<class 'str'> <class 'list'> <class 'dict'>


True

## `__init__` y `self`

`__init__` es un método especial que Python llama **automáticamente** al crear la instancia.
Sirve para inicializar los atributos.

`self` es la instancia que se está creando (o sobre la que se llama el método).

In [2]:
class Observacion:
    def __init__(self, a, b, c):
        self.ciudad = a
        self.temperatura = b
        self.humedad = c
        self.humor = "mala onda"
        

In [5]:
o1 = Observacion("Bariloche", 8.5, 70)
print(o1.ciudad)


Bariloche


In [13]:
print(o1.ciudad, o1.temperatura, o1.humedad)

Bariloche 8.5 70


In [15]:
o2 = Observacion("Neuquén", 21.0, 35)

print(o2.ciudad, o2.temperatura, o2.humedad)

Neuquén 21.0 35


Al escribir `Observacion("Bariloche", 8.5, 70)` Python hace, aproximadamente:

1. crea un objeto vacío,
2. llama `__init__(objeto, "Bariloche", 8.5, 70)` → ese objeto es `self`,
   y asigna los atributos
4. devuelve el objeto.

Por eso `__init__` tiene un parámetro más (`self`) que los argumentos que pasamos.
`self` es solo una convención de nombre, pero **siempre** se usa.

`__init__` no devuelve nada (no lleva `return`).

In [ ]:
# Ahora olvidar un dato falla en el momento de crear el objeto:
try:
    Observacion("Esquel", 5.0)
except TypeError as e:
    print("Error:", e)

### Valores por defecto y validación

`__init__` es una función común: admite parámetros por defecto y puede validar o convertir.

In [ ]:
class Observacion:
    def __init__(self, ciudad, temperatura, humedad=None):
        self.ciudad = ciudad.strip()          # limpiamos espacios
        self.temperatura = float(temperatura) # aceptamos "8.5" o 8.5
        if humedad is not None and not 0 <= humedad <= 100:
            raise ValueError(f"Humedad fuera de rango: {humedad}")
        self.humedad = humedad

o = Observacion("  Bariloche ", "8.5")
print(repr(o.ciudad), o.temperatura, o.humedad)

## Métodos

Un método es una función definida dentro de la clase. Su primer parámetro es `self`,
así puede leer y modificar los atributos de esa instancia.

In [10]:
class Observacion:
    def __init__(self, ciudad, temperatura, humedad):
        self.ciudad = ciudad
        self.temperatura = temperatura
        self.humedad = humedad

    def temperatura_f(self):
        """Devuelve la temperatura en grados Fahrenheit."""
        return self.temperatura * 9 / 5 + 32

    def es_humeda(self, umbral=80):
        return self.humedad >= umbral

    def corregir_temperatura(self, delta):
        """Modifica el objeto: no devuelve nada."""
        self.temperatura += delta
        

In [16]:
lautaro = Observacion("catriel", 35, 10)

print(lautaro.temperatura_f())

95.0


In [17]:
print(Observacion.temperatura_f(lautaro))

95.0


In [7]:
cadena = "hoy está nublado"
cadena.capitalize()

'Hoy está nublado'

In [8]:
str.capitalize(cadena)

'Hoy está nublado'

In [18]:
o = Observacion("Bariloche", 8.5, 85)
o.temperatura

8.5

In [19]:
o.temperatura_f()
o.

47.3

In [ ]:
print(o.temperatura_f())
print(o.es_humeda(), o.es_humeda(umbral=90))
o.corregir_temperatura(-0.5)
print(o.temperatura)

### ¿Qué es `self` en un método?

`o.temperatura_f()` es lo mismo que `Observacion.temperatura_f(o)`:
Python pasa el objeto de la izquierda del punto como primer argumento.

In [ ]:
print(o.temperatura_f())
print(Observacion.temperatura_f(o))

Lo mismo pasa con métodos que ya conocen: `"hola".upper()` equivale a `str.upper("hola")`.

In [ ]:
str.upper("hola"), "hola".upper()

**Error típico:** olvidar `self` en la definición.

In [ ]:
class Mal:
    def saludar():          # falta self
        return "hola"

try:
    Mal().saludar()
except TypeError as e:
    print("Error:", e)

**Otro error típico:** olvidar `self.` al usar un atributo dentro de un método
(`return temperatura * 9 / 5 + 32` → `NameError`).

## Atributos de clase vs. de instancia

Un atributo definido en el cuerpo de la clase (fuera de los métodos) es **compartido** por todas las instancias.

In [ ]:
class Observacion:
    fuente = "SMN"   # atributo de clase

    def __init__(self, ciudad, temperatura):
        self.ciudad = ciudad          # atributo de instancia
        self.temperatura = temperatura

a = Observacion("Bariloche", 8.5)
b = Observacion("Neuquén", 21.0)
print(a.fuente, b.fuente, Observacion.fuente)

## `__str__` y `__repr__`

¿Qué pasa si imprimimos un objeto?

In [ ]:
o = Observacion("Bariloche", 8.5)
print(o)
o

Python no sabe cómo mostrarlo, así que muestra la clase y una dirección de memoria.
Lo resolvemos con dos métodos especiales:

| Método | Lo usa | Para quién | Idea |
|---|---|---|---|
| `__str__` | `print(obj)`, `str(obj)`, `f"{obj}"` | usuario final | legible, lindo |
| `__repr__` | Jupyter/consola al mostrar un valor, `repr(obj)`, **dentro de listas y diccionarios**, `f"{obj!r}"` | programador | sin ambigüedad; idealmente código que recrea el objeto |

Ambos deben **devolver** (`return`) un string, no imprimirlo.

In [35]:
class Observacion:
    def __init__(self, ciudad, temperatura):
        self.ciudad = ciudad
        self.temperatura = temperatura

    def __str__(self):
        return f"{self.ciudad}: {self.temperatura} °C"

    # def __repr__(self):
    #     return f"Observacion({self.ciudad!r}, {self.temperatura!r})"

In [36]:
o = Observacion("Bariloche", 8.5)

In [37]:
print(o)          # usa __str__

Bariloche: 8.5 °C


In [38]:
o

In [24]:
print([1, 2, 3])

[1, 2, 3]


In [25]:
[1, 2, 3]

[1, 2, 3]

In [ ]:
ej = "hola qué tal\nChau hasta luego"

In [28]:
ej

'hola qué tal\nChau hasta luego'

In [29]:
print(ej)

hola qué tal
Chau hasta luego


In [ ]:
print(str(o))
print(repr(o))
print(f"{o}  |  {o!r}")

### Las colecciones usan `__repr__` de sus elementos

In [ ]:
observaciones = [Observacion("Bariloche", 8.5), Observacion("Neuquén", 21.0)]
print(observaciones)

Por eso conviene definir **siempre** `__repr__`. Si solo hay `__repr__`, `print` lo usa también
(cuando falta `__str__`, Python cae en `__repr__`; al revés no).

In [ ]:
class SoloRepr:
    def __repr__(self):
        return "SoloRepr()"

class SoloStr:
    def __str__(self):
        return "soy SoloStr"

print(SoloRepr(), [SoloRepr()])
print(SoloStr(), [SoloStr()])

### ¿Por qué `!r` en `__repr__`?

`!r` pone comillas a los strings, así el resultado se puede copiar y pegar como código:

In [ ]:
texto = repr(o)
print(texto)
copia = eval(texto)    # solo para mostrar la idea; eval no se usa con datos externos
print(copia, copia is o)

## Juntando todo

Una estación con varias observaciones. Repasamos además `sorted` con `key`.

In [41]:
class Observacion:
    def __init__(self, ciudad, temperatura, viento_kmh):
        self.ciudad = ciudad
        self.temperatura = temperatura
        self.viento_kmh = viento_kmh

    def __str__(self):
        return f"{self.ciudad:<12} {self.temperatura:5.1f} °C {self.viento_kmh:4d} km/h"

    def __repr__(self):
        return f"Observacion({self.ciudad!r}, {self.temperatura!r}, {self.viento_kmh!r})"


class Reporte:
    def __init__(self, fecha):
        self.fecha = fecha
        self.observaciones = []      # cada Reporte tiene SU lista

    def agregar(self, obs):
        self.observaciones.append(obs)

    def mas_calidas(self, n=3):
        return sorted(self.observaciones, key=lambda o: o.temperatura, reverse=True)[:n]

    def __str__(self):
        lineas = [f"Reporte {self.fecha} ({len(self.observaciones)} ciudades)"]
        lineas += [str(o) for o in self.observaciones]
        return "\n".join(lineas)

    def __repr__(self):
        return f"Reporte({self.fecha!r})"


r = Reporte("2026-10-20")
r.agregar(Observacion("Bariloche", 8.5, 25))
r.agregar(Observacion("Neuquén", 21.0, 40))
r.agregar(Observacion("Ushuaia", 3.2, 55))
r.agregar(Observacion("Mendoza", 24.1, 10))
print(r)

Reporte 2026-10-20 (4 ciudades)
Bariloche      8.5 °C   25 km/h
Neuquén       21.0 °C   40 km/h
Ushuaia        3.2 °C   55 km/h
Mendoza       24.1 °C   10 km/h


In [11]:
r.mas_calidas(2)

[Observacion('Mendoza', 24.1, 10), Observacion('Neuquén', 21.0, 40)]

### `key` con `getattr` y `operator.attrgetter`

En vez de una `lambda`, podemos pedir el atributo por **nombre** (un string):

- `getattr(obj, "temperatura")` es lo mismo que `obj.temperatura`;
- `attrgetter("temperatura")` devuelve una *función* que hace eso, lista para pasar como `key`.

In [42]:
o = r.observaciones
print(o)

[Observacion('Bariloche', 8.5, 25), Observacion('Neuquén', 21.0, 40), Observacion('Ushuaia', 3.2, 55), Observacion('Mendoza', 24.1, 10)]


In [47]:
# obs = r.observaciones[0]
obs = Observacion("anasagasti", 15, 89)
print(obs.temperatura, getattr(obs, "temperatura"))

15 15


In [49]:
# con lambda + getattr: el campo puede venir enObservacion('Bariloche', 8.5, 25)Observacion('Bariloche', 8.5, 25)Observacion('Bariloche', 8.5, 25) una variable

sorted(r.observaciones, key=lambda o: getattr(o, "temperatura"))

[Observacion('Ushuaia', 3.2, 55),
 Observacion('Bariloche', 8.5, 25),
 Observacion('Neuquén', 21.0, 40),
 Observacion('Mendoza', 24.1, 10)]

## Ejercicios

1. Agregar a `Observacion` un método `sensacion_termica_simple(self)` que reste 1 °C cada 10 km/h de viento (fórmula de juguete).
2. Escribir `Reporte.mas_ventosas(n)` y luego generalizar ambas en `Reporte.top(campo, n, mayor=True)` usando `attrgetter(campo)` como `key`.
3. Definir una clase `Punto(x, y)` con `__str__` → `(1, 2)`, `__repr__` → `Punto(1, 2)` y un método `distancia(self, otro)`.
4. Modificar `Observacion.__init__` para que reciba el viento como texto `"Norte 3"` y guarde dos atributos: `direccion` y `velocidad`.
5. (Pensar) ¿Qué pasaría si en `Reporte` se define `observaciones = []` como atributo de clase en vez de en `__init__`? Probar con dos reportes.